In [1]:
import pandas as pd
import numpy as np
import pickle

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)

from imblearn.over_sampling import RandomOverSampler

In [2]:
# Basic settings
seed = 42
n_splits = 5

kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)

# Hyperparameter
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 10],
    "gamma": [0, 0.1, 0.5, 1]
}

In [3]:
def evaluate_xgboost(model_number, upsample=False):
    """
    Evaluate XGBoost using hyperparameter tuning, cross-validation and test set.
    """

    # Load training and test data
    X_train_full = pd.read_csv(
        f"../data/X_train_{model_number}.csv",
        keep_default_na=False
    )

    y_train_full = pd.read_csv(
        f"../data/y_train_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    X_test = pd.read_csv(
        f"../data/X_test_{model_number}.csv",
        keep_default_na=False
    )

    y_test = pd.read_csv(
        f"../data/y_test_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    # Check
    print("\n")
    print(model_number)

    print("X_train shape:", X_train_full.shape)
    print("X_test shape:", X_test.shape)

    print("\ny_train distribution:")
    print(pd.Series(y_train_full).value_counts())

    print("\ny_test distribution:")
    print(pd.Series(y_test).value_counts())

    # Prepare data
    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_tune, y_tune = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )
    else:
        X_tune = X_train_full
        y_tune = y_train_full

    # Create a base XGBoost model
    base_xgb = XGBClassifier(
        n_estimators=250,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=seed,
        n_jobs=-1
    )

    # Search for better hyperparameters
    random_search = RandomizedSearchCV(
        estimator=base_xgb,
        param_distributions=param_grid,
        n_iter=30,
        scoring="roc_auc",
        cv=kf,
        random_state=seed,
        n_jobs=-1,
        verbose=1
    )

    random_search.fit(X_tune, y_tune)

    best_params = random_search.best_params_

    print("\nBest parameters:")
    print(best_params)

    print("\nBest tuning ROC AUC:")
    print(round(random_search.best_score_, 4))

    # Store cross-validation results
    cv_scores = {
        "fold": [],
        "cv_precision": [],
        "cv_recall": [],
        "cv_roc_auc": [],
        "cv_accuracy": [],
        "cv_f1": []
    }

    # Run cv using the best parameters
    splits = list(kf.split(X_train_full, y_train_full))

    for fold, (train_idx, val_idx) in enumerate(splits, start=1):

        # Split data into training and validation
        X_train = X_train_full.iloc[train_idx]
        y_train = y_train_full[train_idx]

        X_val = X_train_full.iloc[val_idx]
        y_val = y_train_full[val_idx]

        # Upsampling to the training
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(
                X_train,
                y_train
            )

        # Train XGBoost with the selected parameters
        clf = XGBClassifier(
            **best_params,
            n_estimators=250,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1
        )

        clf.fit(X_train, y_train)

        # Predict labels and probabilities
        preds = clf.predict(X_val)
        prob_preds = clf.predict_proba(X_val)[:, 1]

        # Save scores
        cv_scores["fold"].append(fold)
        cv_scores["cv_precision"].append(precision_score(y_val, preds, zero_division=0))
        cv_scores["cv_recall"].append(recall_score(y_val, preds, zero_division=0))
        cv_scores["cv_roc_auc"].append(roc_auc_score(y_val, prob_preds))
        cv_scores["cv_accuracy"].append(accuracy_score(y_val, preds))
        cv_scores["cv_f1"].append(f1_score(y_val, preds, zero_division=0))

    cv_results = pd.DataFrame(cv_scores)
    cv_summary = cv_results.drop(columns=["fold"]).agg(["mean", "sem"])

    print("\nCross-validation results:")
    display(cv_results)

    print("\nCross-validation summary:")
    display(cv_summary)

    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_train_final, y_train_final = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )
    else:
        X_train_final = X_train_full
        y_train_final = y_train_full

    # Train the final model on the full training set
    final_model = XGBClassifier(
        **best_params,
        n_estimators=250,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=seed,
        n_jobs=-1
    )

    final_model.fit(X_train_final, y_train_final)

    # Evaluate the final model on test set
    test_preds = final_model.predict(X_test)
    test_prob_preds = final_model.predict_proba(X_test)[:, 1]
    test_results = pd.DataFrame([{
        "test_precision": precision_score(y_test, test_preds, zero_division=0),
        "test_recall": recall_score(y_test, test_preds, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, test_prob_preds),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_f1": f1_score(y_test, test_preds, zero_division=0)
    }])

    print("\nFinal test results:")
    display(test_results)

    return (
        best_params,
        cv_results,
        cv_summary,
        test_results,
        final_model
    )

In [4]:
# Run
model_settings = {
    "model_1": False,
    "model_1a": True,
    "model_1b": True,
    "model_2": False,
    "model_2a": True,
    "model_2b": True
}

best_params_dict = {}
cv_results_dict = {}
cv_summary_dict = {}
test_results_list = []
final_model_dict = {}

for model_name, use_upsample in model_settings.items():

    (
        best_params,
        cv_results,
        cv_summary,
        test_results,
        final_model
    ) = evaluate_xgboost(
        model_name,
        upsample=use_upsample
    )

    best_params_dict[model_name] = best_params
    cv_results_dict[model_name] = cv_results
    cv_summary_dict[model_name] = cv_summary
    final_model_dict[model_name] = final_model

    test_results["model"] = model_name
    test_results_list.append(test_results)



model_1
X_train shape: (32108, 59)
X_test shape: (8028, 59)

y_train distribution:
1    17321
0    14787
Name: count, dtype: int64

y_test distribution:
1    4337
0    3691
Name: count, dtype: int64
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
{'subsample': 0.8, 'min_child_weight': 3, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.6}

Best tuning ROC AUC:
0.9157

Cross-validation results:


,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.845565,0.861432,0.913034,0.840392,0.853425
1,2,0.852924,0.850462,0.914106,0.840237,0.851691
2,3,0.853109,0.863453,0.918990,0.846154,0.858250
3,4,0.848949,0.863164,0.914309,0.843351,0.855998
4,5,0.860800,0.864030,0.918070,0.851292,0.862412



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.852269,0.860508,0.915702,0.844285,0.856355
sem,0.002548,0.002549,0.001184,0.002062,0.001881



Final test results:


...

,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.867517,0.838125,0.840416,0.789878,0.852567
1,2,0.868109,0.851472,0.850102,0.798561,0.859710
2,3,0.863764,0.841889,0.844562,0.789134,0.852686
3,4,0.867804,0.835729,0.845941,0.788638,0.851464
4,5,0.865501,0.830253,0.844136,0.783428,0.847511



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.866539,0.839493,0.845031,0.789928,0.852788
sem,0.000831,0.003540,0.001563,0.002440,0.001969



Final test results:


,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,0.861422,0.847658,0.841798,0.792022,0.854485


In [5]:
# Save
cv_results_all = pd.concat(cv_results_dict, names=["model", "row"])
cv_summary_all = pd.concat(cv_summary_dict, names=["model", "stat"])

test_summary_all = pd.concat(test_results_list, ignore_index=True)
test_summary_all = test_summary_all[
    [
        "model",
        "test_precision",
        "test_recall",
        "test_roc_auc",
        "test_accuracy",
        "test_f1"
    ]
]

best_params_all = pd.DataFrame(best_params_dict).T

print("Best parameters:")
display(best_params_all)

print("\nCV summaries:")
display(cv_summary_all)

print("\nTest results:")
display(test_summary_all)

best_params_all.to_csv("../data/xgboost_best_params_all.csv")
cv_results_all.to_csv("../data/xgboost_cv_results_all.csv")
cv_summary_all.to_csv("../data/xgboost_cv_summary_all.csv")
test_summary_all.to_csv("../data/xgboost_test_summary_all.csv", index=False)

with open("../data/xgboost_final_models.pkl", "wb") as f:
    pickle.dump(final_model_dict, f)

Best parameters:


,subsample,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree
model_1,0.8,3.0,7.0,0.05,0.5,0.6
model_1a,1.0,10.0,10.0,0.10,0.1,0.8
model_1b,1.0,10.0,10.0,0.10,0.1,0.8
model_2,0.8,3.0,7.0,0.05,0.5,0.6
model_2a,0.8,3.0,7.0,0.05,0.5,0.6
model_2b,1.0,10.0,10.0,0.10,0.1,0.8



CV summaries:


cv_precision  cv_recall  cv_roc_auc  cv_accuracy     cv_f1
model    stat                                                            
model_1  mean      0.852269   0.860508    0.915702     0.844285  0.856355
         sem       0.002548   0.002549    0.001184     0.002062  0.001881
model_1a mean      0.612888   0.653571    0.853606     0.804348  0.632499
         sem       0.007633   0.004189    0.003773     0.003604  0.005239
model_1b mean      0.886704   0.901369    0.888427     0.848871  0.893957
         sem       0.001753   0.003036    0.001600     0.001751  0.001343
model_2  mean      0.805187   0.864086    0.840994     0.775179  0.833592
         sem       0.001534   0.001222    0.001467     0.001413  0.000961
model_2a mean      0.723678   0.721361    0.782604     0.707191  0.722476
         sem       0.002750   0.005256    0.005801     0.002834  0.003170
model_2b mean      0.866539   0.839493    0.845031     0.789928  0.852788
         sem       0.000831   0.003540    0.001563     0.002440  0.001969


Test results:


,model,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,model_1,0.850331,0.859350,0.913756,0.842302,0.854817
1,model_1a,0.586560,0.659411,0.856277,0.789562,0.620856
2,model_1b,0.885810,0.890045,0.888660,0.841437,0.887923
3,model_2,0.795847,0.865620,0.836389,0.768062,0.829268
4,model_2a,0.726198,0.732120,0.791639,0.709936,0.729147
5,model_2b,0.861422,0.847658,0.841798,0.792022,0.854485
